## Environment Setup

Set up the environment. You may refer to [Environment Setup](https://wikidocs.net/257836) for more details.

**[Note]**
- `langchain-opentutorial` is a package that provides a set of easy-to-use environment setup, useful functions and utilities for tutorials. 
- You can checkout the [`langchain-opentutorial`](https://github.com/LangChain-OpenTutorial/langchain-opentutorial-pypi) for more details.

In [9]:
%%capture --no-stderr
!pip install langchain-opentutorial

In [10]:
# Install required packages
from langchain_opentutorial import package

package.install(
    [
        "langchain_core",  # Core functionality of LangChain
        "langchain_community",  # Community-supported integrations
        "langchain_openai",  # OpenAI integration for embeddings and models
        "rank_bm25",  # BM25 ranking algorithm for information retrieval
    ],
    verbose=False,  # Suppress detailed installation logs
    upgrade=False,  # Do not upgrade packages if already installed
)


[notice] A new release of pip is available: 24.0 -> 25.1
[notice] To update, run: pip install --upgrade pip


In [11]:
# Set environment variables
from langchain_opentutorial import set_env

set_env(
    {
        "OPENAI_API_KEY": "",
        "LANGCHAIN_API_KEY": "",
        "LANGCHAIN_TRACING_V2": "true",
        "LANGCHAIN_ENDPOINT": "https://api.smith.langchain.com",
        "LANGCHAIN_PROJECT": "Conversation-With-History",
    }
)

Environment variables have been set successfully.


In [19]:
# Configuration file to manage API keys as environment variables
from dotenv import load_dotenv

# Load API key information
load_dotenv(override=True)

True

## FAISS 벡터 스토어 생성 & Retriever 생성 

EnsembleRetriever 를 이용하여 BM25Retriever와 FAISS 검색기를 결합후 
bm25_retriever, faiss_retriever 의 가중치 할당 

In [21]:
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from langchain.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

# list sample documents
doc_list = [
    "I like apples",
    "I like apple company",
    "I like apple's iphone",
    "Apple is my favorite company",
    "I like apple's ipad",
    "I like apple's macbook",
]

# Initialize the bm25 retriever and faiss retriever.
bm25_retriever = BM25Retriever.from_texts(
    doc_list,
)
bm25_retriever.k = 1  # Set the number of search results for BM25Retriever to 1.

embedding = OpenAIEmbeddings()  # Enable OpenAI embedding.

faiss_vectorstore = FAISS.from_texts(
    doc_list,
    embedding,
)
faiss_retriever = faiss_vectorstore.as_retriever(search_kwargs={"k": 1})

# Initialize the ensemble retriever.
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.7, 0.3],
)

## Query Execution
Perform retrieval for a given query using ensemble_retriever and compare results across retrievers.
- Call the `get_relevant_documents()` method of the `ensemble_retriever` object to retrieve relevant documents.


In [22]:
# Get the search results document.
query = "my favorite fruit is apple"
ensemble_result = ensemble_retriever.invoke(query)
bm25_result = bm25_retriever.invoke(query)
faiss_result = faiss_retriever.invoke(query)

# Output the fetched documents.

# 앙상블 검색기
print("[Ensemble Retriever]")
for doc in ensemble_result:
    print(f"Content: {doc.page_content}")
    print()

'''
[ 출력 결과 ] # 앙상블 검색기는 두개의 검색기를 결합하여 두개의 결과가 출력된다.
[Ensemble Retriever]
Content: Apple is my favorite company
Content: I like apples
'''

# BM25 검색기
print("[BM25 Retriever]")
for doc in bm25_result:
    print(f"Content: {doc.page_content}")
    print()

'''
[ 출력 결과 ]
[BM25 Retriever]
Content: Apple is my favorite company
'''

# FAISS 검색기
print("[FAISS Retriever]")
for doc in faiss_result:
    print(f"Content: {doc.page_content}")
    print()
'''
[ 출력 결과 ]
[FAISS Retriever]
Content: I like apples
'''

[Ensemble Retriever]
Content: Apple is my favorite company

Content: I like apples

[BM25 Retriever]
Content: Apple is my favorite company

[FAISS Retriever]
Content: I like apples



In [23]:
# Get the search results document.
query = "Apple company makes my favorite iphone"
ensemble_result = ensemble_retriever.invoke(query)
bm25_result = bm25_retriever.invoke(query)
faiss_result = faiss_retriever.invoke(query)

# 앙상블 검색기
print("[Ensemble Retriever]")
for doc in ensemble_result:
    print(f"Content: {doc.page_content}")
    print()

'''
[ 출력 결과 ]
[Ensemble Retriever]
Content: Apple is my favorite company

Content: I like apple's iphone

'''
# BM25 검색기
print("[BM25 Retriever]")
for doc in bm25_result:
    print(f"Content: {doc.page_content}")
    print()
'''
[ 출력 결과 ]
[BM25 Retriever]
Content: Apple is my favorite company

'''
# FAISS 검색기
print("[FAISS Retriever]")
for doc in faiss_result:
    print(f"Content: {doc.page_content}")
    print()
'''
[ 출력 결과 ]
[FAISS Retriever]
Content: I like apple's iphone
'''

[Ensemble Retriever]
Content: Apple is my favorite company

Content: I like apple's iphone

[BM25 Retriever]
Content: Apple is my favorite company

[FAISS Retriever]
Content: I like apple's iphone



## Change runtime config

You can also change the properties of a retriever at runtime. This is possible using the `ConfigurableField` class.

- Define the `weights` parameter as a `ConfigurableField` object.
  - Set the field's ID to “ensemble_weights”.


In [24]:
from langchain_core.runnables import ConfigurableField


# 앙상블 검색기 생성
ensemble_retriever = EnsembleRetriever(
    # 검색기 설정 bm25_retriever, faiss_retriever.
    retrievers=[bm25_retriever, faiss_retriever],
).configurable_fields(
    weights=ConfigurableField(
        # Config 고유 ID 값 설정
        id="ensemble_weights",
        # 검색 매개변수 이름 설정
        name="Ensemble Weights",
        # 검색 매개변수 설명 설정 
        description="Ensemble Weights",
    )
)

- Specify the search settings via the `config` parameter when searching.
  - Set the weight of the `ensemble_weights` option to [1, 0] so that **all search results are weighted more heavily toward BM25 retriever**.

In [25]:

# Config 의 설정 가중치 1 : 0 
config = {"configurable": {"ensemble_weights": [1, 0]}}

# config를 매개변수를 사용하여 검색 설정 지정
docs = ensemble_retriever.invoke("my favorite fruit is apple", config=config)
docs  # Print the search result, docs.

[Document(metadata={}, page_content='Apple is my favorite company'),
 Document(id='81c8a71c-cc7c-4619-a866-383f029fad29', metadata={}, page_content='I like apples')]

This time, we want all search results to be weighted **more heavily in favor of the FAISS retriever**.

In [27]:
config = {"configurable": {"ensemble_weights": [0, 1]}}

# Use the config parameter to specify search settings.
docs = ensemble_retriever.invoke("my favorite fruit is apple", config=config)
docs  # Print the search result, docs.

[Document(id='81c8a71c-cc7c-4619-a866-383f029fad29', metadata={}, page_content='I like apples'),
 Document(metadata={}, page_content='Apple is my favorite company')]